# SEQUENCE CLUSTERING using MMSEQS


This notebook collects `.fasta`/`.fna`/`.faa` files from the `viral_data_all/ncbi_dataset/data_subset` directories, merges them into a single comprehensive FASTA file, and uses `mmseqs easy-linclust` to cluster sequences by similarity.

**Note on Scale**: For large datasets (millions of sequences), we use `--split-memory-limit 16G` to handle computations within available RAM (32GB).
**Note on Viral Genomes**: We override `--max-seq-len 10000000` to prevent alignment buffer overflows on Megabase-scale giant viruses.

In [1]:
import os
import glob
import subprocess
import shutil

# 1. Define Paths
data_dir = '../viral_data_all/ncbi_dataset/data_subset/'
output_dir = '../clustering_results/'
combined_fasta = os.path.join(output_dir, 'all_sequences.fasta')
tmp_dir = os.path.join(output_dir, 'tmp_mmseqs')

os.makedirs(output_dir, exist_ok=True)

# 2. Identify the target FASTA files
# finding all files starting with GCA, GCF, ending with _genomic.fna, without 'cds'
file_pattern = '**/GC*_genomic.fna'

fasta_files = [f for f in glob.glob(os.path.join(data_dir, file_pattern), recursive=True) if 'cds' not in os.path.basename(f)]

print(f"Found {len(fasta_files)} FASTA files matching the pattern.")

Found 211307 FASTA files matching the pattern.


In [2]:
from collections import Counter

subfolder_counts = Counter(path.split('data_subset/')[1].split('/')[0] for path in fasta_files)
folders_with_multiple = [folder for folder, count in subfolder_counts.items() if count > 1]
print(folders_with_multiple)


[]


In [3]:
db_path = '../mmseqs_db'

print(f"Creating MMseqs2 database at {db_path} ...")
# Open mmseqs createdb expecting input from stdin
cmd = ["mmseqs", "createdb", "stdin", db_path]
with subprocess.Popen(cmd, stdin=subprocess.PIPE) as process:
    for fpath in fasta_files:
        with open(fpath, 'rb') as f:
            # Efficiently stream each sequence file into mmseqs
            shutil.copyfileobj(f, process.stdin)
    process.stdin.close()
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)

print("MMseqs2 database creation complete!")


Creating MMseqs2 database at ../mmseqs_db ...
createdb stdin ../mmseqs_db 

MMseqs Version:                    	18.8cc5c
Database type                      	0
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	10
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[===================================================================================================	1 Mio. sequences processed
Time for merging to mmseqs_db_h: 0h 0m 0s 314ms
Time for merging to mmseqs_db: 0h 0m 9s 939ms
Database type: Nucleotide
Time for processing: 0h 1m 28s 243ms
MMseqs2 database creation complete!


In [6]:
import subprocess
import os
# Paths from the createdb step
db_path = '../clustering_results/mmseqs_db'
cluster_db = '../clustering_results/clusterRes'
tmp_dir = '../clustering_results/tmp_mmseqs'
os.makedirs(tmp_dir, exist_ok=True)
# 1. Run linclust on the pre-built database
print("Running linclust ...")
subprocess.run([
    "mmseqs", "linclust",
    db_path,
    cluster_db,
    tmp_dir,
    "--min-seq-id", "0.95",
    "-c", "0.95",
    "--cov-mode", "1",
    "--threads", "64",
], check=True)
print("Linclust complete!")
# 2. Export cluster assignments to TSV
print("Exporting cluster TSV ...")
subprocess.run([
    "mmseqs", "createtsv",
    db_path, db_path, cluster_db,
    "../clustering_results/clusterRes_cluster.tsv",
], check=True)
# 3. Extract representative sequences to FASTA
print("Exporting representative sequences ...")
subprocess.run([
    "mmseqs", "result2repseq",
    db_path, cluster_db,
    "../clustering_results/clusterRes_rep",
], check=True)
subprocess.run([
    "mmseqs", "result2flat",
    db_path, db_path,
    "../clustering_results/clusterRes_rep",
    "../clustering_results/clusterRes_rep_seq.fasta",
    "--use-fasta-header",
], check=True)
# 4. Extract all clustered sequences to FASTA
print("Exporting all clustered sequences ...")
subprocess.run([
    "mmseqs", "createseqfiledb",
    db_path, cluster_db,
    "../clustering_results/clusterRes_all_seqs_db",
], check=True)
subprocess.run([
    "mmseqs", "result2flat",
    db_path, db_path,
    "../clustering_results/clusterRes_all_seqs_db",
    "../clustering_results/clusterRes_all_seqs.fasta",
], check=True)
print("Done! Output files:")
print("  - clusterRes_cluster.tsv       (cluster membership)")
print("  - clusterRes_rep_seq.fasta     (representative sequences)")
print("  - clusterRes_all_seqs.fasta    (all sequences grouped by cluster)")

Running linclust ...
linclust ../clustering_results/mmseqs_db ../clustering_results/clusterRes ../clustering_results/tmp_mmseqs --min-seq-id 0.95 -c 0.95 --cov-mode 1 --threads 64 

MMseqs Version:                     	18.8cc5c
Cluster mode                        	0
Max connected component depth       	1000
Similarity type                     	2
Threads                             	64
Compressed                          	0
Verbosity                           	3
Weight file name                    	
Cluster Weight threshold            	0.9
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Add backtrace                       	false
Alignment mode                      	2
Alignment mode                      	0
Allow wrapped scoring               	false
E-value threshold                   	0.001
Seq. id. threshold                  	0.95
Min alignment length                	0
Seq. id. mode                       	0
Alternative alignments              	0
Coverage thresho

# CHECKING

### Inspecting Results
Output files in `../clustering_results/`:
1. `clusterRes_cluster.tsv` (mappings)
2. `clusterRes_rep_seq.fasta` (representatives)

In [8]:
import pandas as pd
cluster_tsv = os.path.join(output_dir, 'clusterRes_cluster.tsv')

if os.path.exists(cluster_tsv):
    cluster_df = pd.read_csv(cluster_tsv, sep='\t', names=['Representative', 'Member'])
    print("Top 10 Largest Clusters:")
    print(cluster_df['Representative'].value_counts().head(10))
else:
    print(f"{cluster_tsv} not found.")

Top 10 Largest Clusters:
Representative
KY697335.1    48298
KF541239.1    48288
JX138513.1    48286
PV570294.1    48180
MW888331.1    47904
PV570297.1    47630
PV570293.1    46570
PV570315.1    45984
CY054641.1    38693
MN560979.1    35977
Name: count, dtype: int64


In [12]:
cluster_df['Representative'].value_counts()

Representative
KY697335.1    48298
KF541239.1    48288
JX138513.1    48286
PV570294.1    48180
MW888331.1    47904
              ...  
PV388414.1        1
PX813953.1        1
EU581825.1        1
OP890528.1        1
OQ469091.1        1
Name: count, Length: 50057, dtype: int64